# BT4012 Competition — Ensemble of the Tuned Models

Continues from `03_supervised_learning.ipynb`, which tuned XGBoost, LightGBM, Random Forest and the MLP and chose XGBoost as the final model. Different models make different mistakes, so averaging them can rank transactions better than any single one. This notebook tests that, with the same setup as `03`:

- the same two validation splits (adjacent: train `time_step` 1–30, validate 31–35; gap-matched: train 1–20, validate 30–35) and the same sample weights;
- **rank averaging**: each model's predictions are converted to percentile ranks before averaging, because ROC-AUC only scores the ordering and the models' probabilities sit on different scales;
- every equal-weight combination that includes the final XGBoost is tested (7 blends), and each is compared with XGBoost alone on both splits.

Each blend's test predictions are saved to `data/submissions/experiments/`; `data/submissions/final_submission.csv` (XGBoost, from `03`) is not touched.


In [1]:
import itertools
import json
import warnings

from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier

## Setup

Data, splits and sample weights as in `03_supervised_learning.ipynb`; the tuned settings come from `data/processed/tuned_models.json`, written by `03`.

In [2]:
train_reduced = pd.read_csv("data/processed/train_reduced.csv")
test_reduced = pd.read_csv("data/processed/test_reduced.csv")
feat_cols = [c for c in train_reduced.columns
             if c not in ("txId", "time_step", "label", "label_propagated", "label_propagated_confidence")]
labeled = train_reduced[train_reduced["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

SPLITS = {
    "adjacent": (labeled[labeled["time_step"] <= 30], labeled[labeled["time_step"].between(31, 35)]),
    "gap-matched": (labeled[labeled["time_step"] <= 20], labeled[labeled["time_step"].between(30, 35)]),
}

tuned = json.load(open("data/processed/tuned_models.json"))
PARAMS = tuned["models"]
if "hidden_layer_sizes" in PARAMS["MLP"]:
    PARAMS["MLP"]["hidden_layer_sizes"] = tuple(PARAMS["MLP"]["hidden_layer_sizes"])
HALF_LIFE = {name: tuned["half_life"] for name in PARAMS}
HALF_LIFE["XGBoost"] = tuned["final_half_life"]
for name, params in PARAMS.items():
    print(f"{name}: {params or 'defaults'} | half-life {HALF_LIFE[name]}")


def sample_weight(rows, half_life):
    y = rows["label"]
    weight = y.map({0: 1.0, 1: (y == 0).sum() / (y == 1).sum()})
    if half_life is not None:
        weight = weight * 0.5 ** ((rows["time_step"].max() - rows["time_step"]) / half_life)
    return weight.to_numpy()


def fit_predict(name, tr, rows_to_score):
    weight = sample_weight(tr, HALF_LIFE[name])
    params = PARAMS[name]
    if name == "XGBoost":
        model = XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1, **params)
    elif name == "LightGBM":
        model = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, **params)
    elif name == "Random Forest":
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    else:  # MLP: quantile-scaled inputs, normalized weights, averaged over 3 seeds
        transform = QuantileTransformer(output_distribution="normal", random_state=0).fit(tr[feat_cols])
        weight = weight / weight.mean()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            probs = [MLPClassifier(random_state=seed, **params)
                     .fit(transform.transform(tr[feat_cols]), tr["label"], sample_weight=weight)
                     .predict_proba(transform.transform(rows_to_score[feat_cols]))[:, 1] for seed in range(3)]
        return np.mean(probs, axis=0)
    return model.fit(tr[feat_cols], tr["label"], sample_weight=weight).predict_proba(rows_to_score[feat_cols])[:, 1]


def rank_average(prediction_list):
    return np.mean([rankdata(p) / len(p) for p in prediction_list], axis=0)

XGBoost: {'max_depth': 4, 'learning_rate': 0.2, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1, 'n_estimators': 300} | half-life 10
LightGBM: {'n_estimators': 300, 'num_leaves': 31, 'learning_rate': 0.2, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.7, 'reg_lambda': 1} | half-life 10
Random Forest: defaults | half-life 10
MLP: {'hidden_layer_sizes': (64,), 'alpha': 0.0001, 'max_iter': 200} | half-life 10


## Validation predictions of each tuned model

Each model is trained on each split's training rows and predicts that split's validation rows. The rank correlation shows how differently two models order transactions: a blend can only help where they disagree.

In [3]:
MODELS = ["XGBoost", "LightGBM", "MLP", "Random Forest"]
val_preds = {name: {split: fit_predict(name, tr, va) for split, (tr, va) in SPLITS.items()} for name in MODELS}

single = pd.DataFrame({name: {split: roc_auc_score(SPLITS[split][1]["label"], val_preds[name][split]) for split in SPLITS}
                       for name in MODELS}).T
single["average"] = single.mean(axis=1)
print(single.round(4))

va_gap = SPLITS["gap-matched"][1]
corr = pd.DataFrame({a: {b: pd.Series(val_preds[a]["gap-matched"]).corr(pd.Series(val_preds[b]["gap-matched"]), method="spearman")
                         for b in MODELS} for a in MODELS})
print("\nrank correlation between models (gap-matched split):")
corr.round(3)

               adjacent  gap-matched  average
XGBoost          0.9979       0.9001   0.9490
LightGBM         0.9978       0.8964   0.9471
MLP              0.9572       0.9312   0.9442
Random Forest    0.9966       0.8332   0.9149

rank correlation between models (gap-matched split):


,XGBoost,LightGBM,MLP,Random Forest
XGBoost,1.000,0.864,0.667,0.715
LightGBM,0.864,1.000,0.686,0.792
MLP,0.667,0.686,1.000,0.577
Random Forest,0.715,0.792,0.577,1.000


## Blends

Every equal-weight rank average of XGBoost with one, two or all three of the other tuned models.

In [4]:
others = [m for m in MODELS if m != "XGBoost"]
blends = {}
for k in range(1, len(others) + 1):
    for combo in itertools.combinations(others, k):
        members = ("XGBoost",) + combo
        blends[" + ".join(members)] = members

reference = single.loc["XGBoost"]
blend_rows = {"XGBoost alone (final model)": {**reference.to_dict(), "higher on both": False}}  # "higher on both" shown for reference
for label, members in blends.items():
    scores = {split: roc_auc_score(SPLITS[split][1]["label"], rank_average([val_preds[m][split] for m in members]))
              for split in SPLITS}
    scores["average"] = (scores["adjacent"] + scores["gap-matched"]) / 2
    scores["higher on both"] = scores["adjacent"] > reference["adjacent"] and scores["gap-matched"] > reference["gap-matched"]
    blend_rows[label] = scores

blend_table = pd.DataFrame(blend_rows).T.sort_values("average", ascending=False)
blend_table

,adjacent,gap-matched,average,higher on both
XGBoost + MLP,0.985065,0.924578,0.954822,False
XGBoost + LightGBM + MLP,0.991204,0.917403,0.954303,False
XGBoost + LightGBM,0.998089,0.900584,0.949337,True
XGBoost alone (final model),0.997928,0.900144,0.949036,False
XGBoost + LightGBM + MLP + Random Forest,0.994101,0.894958,0.944529,False
XGBoost + MLP + Random Forest,0.991599,0.891962,0.941781,False
XGBoost + LightGBM + Random Forest,0.998126,0.876109,0.937117,False
XGBoost + Random Forest,0.997995,0.861796,0.929895,False


## Test predictions

Every tuned model is retrained on all labeled rows and predicts the test set; each blend is saved to `data/submissions/experiments/`.


In [5]:
test_preds = {name: fit_predict(name, labeled, test_reduced) for name in MODELS}

for label, members in blends.items():
    filename = "submission_ensemble_" + "_".join(m.split()[0].lower() for m in members) + ".csv"
    target = rank_average([test_preds[m] for m in members])
    pd.DataFrame({"index": np.arange(len(test_reduced)), "target": target}).to_csv(
        f"data/submissions/experiments/{filename}", index=False)
print(f"saved {len(blends)} blends to data/submissions/experiments/")


saved 7 blends to data/submissions/experiments/


## Outcome

On validation, two blends looked at least as good as XGBoost alone: **XGBoost + LightGBM** was higher on both splits by a tiny margin (+0.0002 adjacent, +0.0004 gap-matched), and **XGBoost + MLP** had the best average, driven by the gap-matched split (0.925 vs. 0.900) while losing on the adjacent split (0.985 vs. 0.998). Both were submitted and both scored **below XGBoost alone** on the public leaderboard:

| Submission | Leaderboard ROC-AUC |
|---|---|
| XGBoost alone (final model, `03`) | **0.94936** |
| XGBoost + LightGBM | 0.94654 |
| XGBoost + MLP | 0.93952 |

The gains on validation were either within noise or came from one split only, and they did not carry over to the later test period. **The final model stays the single tuned XGBoost.**
